## Notebook 2: Model training

#### This notebook is the first of two notebooks pertaining the empirical analysis of this thesis. Notebook 1 includes the data preperation process. Notebook 2 includes the training of the models and the application of the in- and post-processors. 

## Bachelor´s Thesis 
#### for acquiring the degree of Bachelor of Science (B.Sc.) at the School of Business and Economics of Humboldt-Universität zu Berlin

### Topic: Algorithmic Fairness at the Intersection of Gender and Age - Evaluating In- and Post-Processing Methods on Credit Risk Data


#### Name: Celina Gina Kaliman
#### Student number: 625187
#### E-Mail address: celina.gina.kaliman@student.hu-berlin.de
#### First Examiner: Prof. Dr. Lessmann
#### Second Examiner: Prof. Dr. Mendling

##### In order to recreate the same running environment, run this command in your terminal: pip install -r requirements.txt

In [1]:
#installing the necessary packages
%pip install fairlearn xgboost scikit-learn pandas numpy openpyxl ucimlrepo jupyter matplotlib

Note: you may need to restart the kernel to use updated packages.


In [2]:
#importing the necessary libraries 
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score
from fairlearn.metrics import MetricFrame, demographic_parity_difference, equalized_odds_difference
from sklearn.metrics import precision_score
from fairlearn.reductions import ExponentiatedGradient, GridSearch, DemographicParity, EqualizedOdds
from sklearn.base import BaseEstimator, ClassifierMixin
from fairlearn.postprocessing import ThresholdOptimizer

In [3]:
#Fetch the data saved from Notebook 1 from the folder "data/Output_Notebook1"
output_notebook1 = "data/Output_Notebook1"

X_train = pd.read_csv(f"{output_notebook1}/X_train.csv")
X_test = pd.read_csv(f"{output_notebook1}/X_test.csv")
X_train_scaled = pd.read_csv(f"{output_notebook1}/X_train_scaled.csv")
X_test_scaled = pd.read_csv(f"{output_notebook1}/X_test_scaled.csv")

y_train = pd.read_csv(f"{output_notebook1}/y_train.csv").squeeze("columns")
y_test = pd.read_csv(f"{output_notebook1}/y_test.csv").squeeze("columns")

sens_train = pd.read_csv(f"{output_notebook1}/sens_train.csv").squeeze("columns")
sens_test = pd.read_csv(f"{output_notebook1}/sens_test.csv").squeeze("columns")
gender_train = pd.read_csv(f"{output_notebook1}/gender_train.csv").squeeze("columns")
gender_test = pd.read_csv(f"{output_notebook1}/gender_test.csv").squeeze("columns")
age_train = pd.read_csv(f"{output_notebook1}/age_train.csv").squeeze("columns")
age_test = pd.read_csv(f"{output_notebook1}/age_test.csv").squeeze("columns")

sens_train.name = "intersection"
sens_test.name = "intersection"
gender_train.name = "gender"
gender_test.name = "gender"
age_train.name = "age_group"
age_test.name = "age_group"

config = {}
with open(f"{output_notebook1}/config.txt") as f:
    for line in f:
        key, _, value = line.strip().partition("=")
        config[key] = value

random_seed = int(config["random_seed"])


print(f"Successfully loaded prepared data. random_seed={random_seed}")
print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")
print(f"Y_train shape: {y_train.shape}, Y_test shape: {y_test.shape}")

Successfully loaded prepared data. random_seed=222
X_train shape: (20975, 19), X_test shape: (8990, 19)
Y_train shape: (20975,), Y_test shape: (8990,)


### Baseline model training

In [4]:
#Implementing a convergence check for the Logistic Regression model to ensure that the model converges properly

for test_iter in [100, 500, 1000, 2000, 5000, 10000]:
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always")
        test_model = LogisticRegression(max_iter=test_iter, random_state=random_seed).fit(X_train_scaled, y_train)
        converged = not any(issubclass(warning.category, ConvergenceWarning) for warning in w)
        print(f"max_iter={test_iter}: {'Converged cleanly' if converged else 'Does not converge cleanly'}")

max_iter=100: Converged cleanly
max_iter=500: Converged cleanly
max_iter=1000: Converged cleanly
max_iter=2000: Converged cleanly
max_iter=5000: Converged cleanly
max_iter=10000: Converged cleanly


In [5]:
#the convergence check shows that the model converges cleanly at as low as max_iter=100. For safety, the max_inter was set to 1000

#three models unbalanced training
models = {
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=random_seed).fit(X_train_scaled, y_train),
    "RandomForest": RandomForestClassifier(random_state=random_seed).fit(X_train, y_train),
    "XGBoost": XGBClassifier(eval_metric="logloss", random_state=random_seed).fit(X_train, y_train),
    }

#three models balanced traning in order to address identified class imbalance problem.
#Logistic regression and RandomForest are balanced through the use of class weights, while XGBoost is balanced through the use of scale_pos_weight parameter.
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum() 

models_balanced = {
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=random_seed, class_weight="balanced").fit(X_train_scaled, y_train),
    "RandomForest": RandomForestClassifier(random_state=random_seed, class_weight="balanced").fit(X_train, y_train),
    "XGBoost": XGBClassifier(eval_metric="logloss", random_state=random_seed, scale_pos_weight=scale_pos_weight).fit(X_train, y_train),
}

print("Trained ", list(models.keys()), "successfully")

Trained  ['LogisticRegression', 'RandomForest', 'XGBoost'] successfully


## Applying the In-Processors 

### Setting up the basis before fitting ExponentiatedGradient and GridSearch

In [6]:
#Fitting is done on three models
#The sensitive variants the models are fitted on are: intersection, gender and age 
#Fitting of the in-processors on two constraints: demographic parity and equalized odds. Predicitive parity is not used as a constraints because it is not supported by fairlearn.

np.random.seed(random_seed)

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum() 

base_estimators = {
    "LogisticRegression": lambda: LogisticRegression(max_iter=1000, random_state=random_seed, class_weight="balanced"),
    "RandomForest": lambda: RandomForestClassifier(random_state=random_seed, class_weight="balanced"),
    "XGBoost": lambda: XGBClassifier(eval_metric="logloss", random_state=random_seed, scale_pos_weight=scale_pos_weight),
}

sensitive_variants_train = {
    "intersection": sens_train.to_numpy(),
    "gender_only": gender_train.to_numpy(),
    "age_only": age_train.to_numpy(),
}

X_train_by_model = {
    "LogisticRegression": X_train_scaled,
    "RandomForest": X_train,
    "XGBoost": X_train,
}

constraint_variants = {
    "demographic_parity": lambda: DemographicParity(),
    "equalized_odds": lambda: EqualizedOdds(),
}

#### 1. In-processor: ExponentiatedGradient

In [7]:
#Fitting ExponentiatedGradient a total of 18 times
expgrad_models = {}

for model_name, estimator_fn in base_estimators.items():
    X_for_this_model = X_train_by_model[model_name]
    for constrained_on, sens_col in sensitive_variants_train.items():
        for constraint_name, constraint_fn in constraint_variants.items():
            print(f"Fitting ExponentiatedGradient + {model_name}, constrained on {constrained_on}, target={constraint_name} in progess.")
            eg = ExponentiatedGradient(estimator=estimator_fn(), constraints=constraint_fn())
            eg.fit(X_for_this_model, y_train, sensitive_features=sens_col)
            expgrad_models[(model_name, constrained_on, constraint_name)] = eg


print("Done.")

Fitting ExponentiatedGradient + LogisticRegression, constrained on intersection, target=demographic_parity in progess.
Fitting ExponentiatedGradient + LogisticRegression, constrained on intersection, target=equalized_odds in progess.
Fitting ExponentiatedGradient + LogisticRegression, constrained on gender_only, target=demographic_parity in progess.
Fitting ExponentiatedGradient + LogisticRegression, constrained on gender_only, target=equalized_odds in progess.
Fitting ExponentiatedGradient + LogisticRegression, constrained on age_only, target=demographic_parity in progess.
Fitting ExponentiatedGradient + LogisticRegression, constrained on age_only, target=equalized_odds in progess.
Fitting ExponentiatedGradient + RandomForest, constrained on intersection, target=demographic_parity in progess.
Fitting ExponentiatedGradient + RandomForest, constrained on intersection, target=equalized_odds in progess.
Fitting ExponentiatedGradient + RandomForest, constrained on gender_only, target=demog

#### 2. In-processor: GridSearch

In [8]:
#GridSearch is fitted with 64 grid points as recommended by the warning message given if the amount od grid points is set lower

#Fitting GridSearch a total of 18 times 
gridsearch_models = {}

for model_name, estimator_fn in base_estimators.items():
    X_for_this_model = X_train_by_model[model_name]
    for constrained_on, sens_col in sensitive_variants_train.items():
        for constraint_name, constraint_fn in constraint_variants.items():
            print(f"Fitting GridSearch + {model_name}, constrained on {constrained_on}, target={constraint_name} in progress.")
            gs = GridSearch(estimator=estimator_fn(), constraints=constraint_fn(), grid_size=64)
            gs.fit(X_for_this_model, y_train, sensitive_features=sens_col)
            gridsearch_models[(model_name, constrained_on, constraint_name)] = gs

print("Done.")

Fitting GridSearch + LogisticRegression, constrained on intersection, target=demographic_parity in progress.


The grid has 6 dimensions. It is not recommended to use more than 4, otherwise a prohibitively large grid size is required to explore the space thoroughly. For such cases consider using ExponentiatedGradient from the fairlearn.reductions module.


Fitting GridSearch + LogisticRegression, constrained on intersection, target=equalized_odds in progress.
Fitting GridSearch + LogisticRegression, constrained on gender_only, target=demographic_parity in progress.
Fitting GridSearch + LogisticRegression, constrained on gender_only, target=equalized_odds in progress.
Fitting GridSearch + LogisticRegression, constrained on age_only, target=demographic_parity in progress.
Fitting GridSearch + LogisticRegression, constrained on age_only, target=equalized_odds in progress.
Fitting GridSearch + RandomForest, constrained on intersection, target=demographic_parity in progress.


The grid has 6 dimensions. It is not recommended to use more than 4, otherwise a prohibitively large grid size is required to explore the space thoroughly. For such cases consider using ExponentiatedGradient from the fairlearn.reductions module.


Fitting GridSearch + RandomForest, constrained on intersection, target=equalized_odds in progress.
Fitting GridSearch + RandomForest, constrained on gender_only, target=demographic_parity in progress.
Fitting GridSearch + RandomForest, constrained on gender_only, target=equalized_odds in progress.
Fitting GridSearch + RandomForest, constrained on age_only, target=demographic_parity in progress.
Fitting GridSearch + RandomForest, constrained on age_only, target=equalized_odds in progress.
Fitting GridSearch + XGBoost, constrained on intersection, target=demographic_parity in progress.


The grid has 6 dimensions. It is not recommended to use more than 4, otherwise a prohibitively large grid size is required to explore the space thoroughly. For such cases consider using ExponentiatedGradient from the fairlearn.reductions module.


Fitting GridSearch + XGBoost, constrained on intersection, target=equalized_odds in progress.
Fitting GridSearch + XGBoost, constrained on gender_only, target=demographic_parity in progress.
Fitting GridSearch + XGBoost, constrained on gender_only, target=equalized_odds in progress.
Fitting GridSearch + XGBoost, constrained on age_only, target=demographic_parity in progress.
Fitting GridSearch + XGBoost, constrained on age_only, target=equalized_odds in progress.
Done.


### Applying the Post-Processor: ThresholdOptimizer 

In [9]:
#Wrapping the XGBoost model to ensure that the output of predict_proba is always data type float64
class Float64Wrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, fitted_model=None):
        self.fitted_model = fitted_model

    def fit(self, X, y=None, **kwargs):
        self.classes_ = self.fitted_model.classes_
        return self

    def predict(self, X):
        return self.fitted_model.predict(X)

    def predict_proba(self, X):
        return self.fitted_model.predict_proba(X).astype("float64")

In [10]:
#Fitting ThresholdOptimizer

threshold_models = {} 

X_train_by_model = {
    "LogisticRegression": X_train_scaled,
    "RandomForest": X_train,
    "XGBoost": X_train,
}

for model_name, model in models_balanced.items():
    X_for_this_model = X_train_by_model[model_name]
    estimator_for_threshold = Float64Wrapper(model) if model_name == "XGBoost" else model
    for constraint in ["demographic_parity", "equalized_odds"]:
        for constrained_on, sens_col in sensitive_variants_train.items():
            key = (model_name, constraint, constrained_on)
            to = ThresholdOptimizer(estimator=estimator_for_threshold, constraints=constraint, prefit=True, predict_method="predict_proba")
            to.fit(X_for_this_model, y_train, sensitive_features=sens_col)
            threshold_models[key] = to

print(f"Fitted {len(threshold_models)} ThresholdOptimizer variants.")

/Users/celinakaliman/Bachelorarbeit/Bachelorarbeit/venv/lib/python3.13/site-packages/fairlearn/postprocessing/_threshold_optimizer.py:329: UserWarning: The value of `prefit` is `True`, but `check_is_fitted` raised `NotFittedError` on the base estimator.

If the provided base estimator has been fitted, this could mean that (1) its implementation does not conform to the sklearn estimator API, or (2) the enclosing ThresholdOptimizer has been cloned (for instance by `sklearn.model_selection.cross_validate`).

In case (1), please file an issue with the base estimator developers, but continue to use the enclosing ThresholdOptimizer with `prefit=True`. In case (2), please use `prefit=False`.
  warn(BASE_ESTIMATOR_NOT_FITTED_WARNING.format(type(self).__name__))
/Users/celinakaliman/Bachelorarbeit/Bachelorarbeit/venv/lib/python3.13/site-packages/fairlearn/postprocessing/_interpolated_thresholder.py:102: UserWarning: The value of `prefit` is `True`, but `check_is_fitted` raised `NotFittedError` 

Fitted 18 ThresholdOptimizer variants.


/Users/celinakaliman/Bachelorarbeit/Bachelorarbeit/venv/lib/python3.13/site-packages/fairlearn/postprocessing/_interpolated_thresholder.py:102: UserWarning: The value of `prefit` is `True`, but `check_is_fitted` raised `NotFittedError` on the base estimator.

If the provided base estimator has been fitted, this could mean that (1) its implementation does not conform to the sklearn estimator API, or (2) the enclosing InterpolatedThresholder has been cloned (for instance by `sklearn.model_selection.cross_validate`).

In case (1), please file an issue with the base estimator developers, but continue to use the enclosing InterpolatedThresholder with `prefit=True`. In case (2), please use `prefit=False`.
  warn(BASE_ESTIMATOR_NOT_FITTED_WARNING.format(type(self).__name__))


## Building the results tables 

#### 1. Results for the fairness scores and accuracy for the baseline, the in-processors and the post-processor for the intersectional, gender and age_only variables

In [11]:
def predictive_parity_difference(y_true, y_pred, sensitive_features):
    mf = MetricFrame(
        metrics=lambda yt, yp: precision_score(yt, yp, zero_division=np.nan),
        y_true=y_true, y_pred=y_pred, sensitive_features=sensitive_features
    )
    return mf.difference()

X_test_by_model = {
    "LogisticRegression": X_test_scaled,
    "RandomForest": X_test,
    "XGBoost": X_test,
}

full_attribute_levels = {
    "gender": gender_test,
    "age_group": age_test,
    "intersection": sens_test,
}

sens_test_by_level = {
    "intersection": sens_test.to_numpy(),
    "gender_only": gender_test.to_numpy(),
    "age_only": age_test.to_numpy(),
}

def compute_row(preds, sens_col, model_name, processor, constrained_on, targeted_constraint, attr_name):
    dpd = demographic_parity_difference(y_test, preds, sensitive_features=sens_col)
    eod = equalized_odds_difference(y_test, preds, sensitive_features=sens_col)
    ppd = predictive_parity_difference(y_test, preds, sensitive_features=sens_col)

    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds)

    return {
        "Processor": processor, "Model": model_name,
        "ConstrainedOn": constrained_on, "TargetedConstraint": targeted_constraint,
        "Attribute": attr_name,
        "DemographicParityDiff": round(dpd, 4),
        "EqualizedOddsDiff": round(eod, 4),
        "PredictiveParityDiff": round(ppd, 4) if not pd.isna(ppd) else np.nan,
        "Accuracy": round(acc, 4),
        "F1_Score": round(f1, 4),
    }

results_rows = []

#Balanced baseline
for model_name, model in models_balanced.items():
    preds = model.predict(X_test_by_model[model_name])
    for attr_name, sens_col in full_attribute_levels.items():
        results_rows.append(compute_row(preds, sens_col, model_name, "Baseline_Balanced", "n/a", "n/a", attr_name))

#ExponentiatedGradient
for (model_name, constrained_on, constraint_name), model in expgrad_models.items():
    preds = model.predict(X_test_by_model[model_name], random_state=random_seed)
    for attr_name, sens_col in full_attribute_levels.items():
        results_rows.append(compute_row(preds, sens_col, model_name, "ExponentiatedGradient", constrained_on, constraint_name, attr_name))

#GridSearch
for (model_name, constrained_on, constraint_name), model in gridsearch_models.items():
    preds = model.predict(X_test_by_model[model_name])
    for attr_name, sens_col in full_attribute_levels.items():
        results_rows.append(compute_row(preds, sens_col, model_name, "GridSearch", constrained_on, constraint_name, attr_name))

#ThresholdOptimizer
for (model_name, constraint, constrained_on), to_model in threshold_models.items():
    preds = to_model.predict(X_test_by_model[model_name], sensitive_features=sens_test_by_level[constrained_on],
                             random_state=random_seed)
    for attr_name, sens_col in full_attribute_levels.items():
        results_rows.append(compute_row(preds, sens_col, model_name, "ThresholdOptimizer", constrained_on, constraint, attr_name))

results_table = pd.DataFrame(results_rows)

results_table = results_table.sort_values(["Attribute", "Processor", "Model", "ConstrainedOn"])

os.makedirs("results", exist_ok=True)
results_table.to_excel("results/entire_results_table.xlsx", index=False)

print(f"Total rows: {len(results_table)}")
print("Saved to results/entire_results_table.xlsx")
results_table

Total rows: 171
Saved to results/entire_results_table.xlsx


,Processor,Model,ConstrainedOn,TargetedConstraint,Attribute,DemographicParityDiff,EqualizedOddsDiff,PredictiveParityDiff,Accuracy,F1_Score
1,Baseline_Balanced,LogisticRegression,n/a,n/a,age_group,0.2808,0.3010,0.0335,0.6800,0.4734
4,Baseline_Balanced,RandomForest,n/a,n/a,age_group,0.0945,0.1095,0.0498,0.7949,0.5265
7,Baseline_Balanced,XGBoost,n/a,n/a,age_group,0.1104,0.0896,0.0170,0.7633,0.5216
22,ExponentiatedGradient,LogisticRegression,age_only,demographic_parity,age_group,0.0631,0.0791,0.0958,0.7309,0.4754
25,ExponentiatedGradient,LogisticRegression,age_only,equalized_odds,age_group,0.0302,0.0218,0.0889,0.6742,0.4349
...,...,...,...,...,...,...,...,...,...,...
170,ThresholdOptimizer,XGBoost,age_only,equalized_odds,intersection,0.0878,0.1365,0.1503,0.7957,0.4974
158,ThresholdOptimizer,XGBoost,gender_only,demographic_parity,intersection,0.0890,0.1573,0.0976,0.7970,0.5098
167,ThresholdOptimizer,XGBoost,gender_only,equalized_odds,intersection,0.1284,0.1403,0.0782,0.7958,0.5078
155,ThresholdOptimizer,XGBoost,intersection,demographic_parity,intersection,0.0353,0.0993,0.1800,0.7960,0.5166


#### 2. Supporting table with the results for each intersectional subgroups

In [12]:
groups = ["female_under25", "female_25plus", "male_under25", "male_25plus"]
intersection_values = sens_test.to_numpy()
y_true = y_test.to_numpy().astype(int)

#Predictions of all the fitted models
predictions = {}

for model_name, model in models_balanced.items():
    predictions[("Baseline_Balanced", model_name, "n/a", "n/a")] = np.asarray(
        model.predict(X_test_by_model[model_name])).astype(int)

for (model_name, constrained_on, constraint_name), model in expgrad_models.items():
    predictions[("ExponentiatedGradient", model_name, constrained_on, constraint_name)] = np.asarray(
        model.predict(X_test_by_model[model_name], random_state=random_seed)).astype(int)

for (model_name, constrained_on, constraint_name), model in gridsearch_models.items():
    predictions[("GridSearch", model_name, constrained_on, constraint_name)] = np.asarray(
        model.predict(X_test_by_model[model_name])).astype(int)

for (model_name, constraint, constrained_on), to_model in threshold_models.items():
    predictions[("ThresholdOptimizer", model_name, constrained_on, constraint)] = np.asarray(
        to_model.predict(X_test_by_model[model_name],
                         sensitive_features=sens_test_by_level[constrained_on],
                         random_state=random_seed)).astype(int)

#Rates for each subgroup
def group_rates(y, p):
    return {
        "SelectionRate": p.mean(),                                        
        "TPR": p[y == 1].mean() if (y == 1).any() else np.nan,         
        "FPR": p[y == 0].mean() if (y == 0).any() else np.nan,            
        "Precision": y[p == 1].mean() if (p == 1).any() else np.nan,      
        "Accuracy": accuracy_score(y, p),
        "F1_Score": f1_score(y, p, zero_division=0),
    }

rate_names = ["SelectionRate", "TPR", "FPR", "Precision", "Accuracy", "F1_Score"]

#Baseline values per model and subgroup
baseline_rates = {}
for model_name in models_balanced:
    p_base = predictions[("Baseline_Balanced", model_name, "n/a", "n/a")]
    for g in groups:
        mask = intersection_values == g
        baseline_rates[(model_name, g)] = group_rates(y_true[mask], p_base[mask])

#Building the results table
group_rows = []
for (processor, model_name, constrained_on, constraint), p in predictions.items():
    for g in groups:
        mask = intersection_values == g
        rates = group_rates(y_true[mask], p[mask])
        row = {"Processor": processor, "Model": model_name,
               "ConstrainedOn": constrained_on, "TargetedConstraint": constraint,
               "Group": g, "n": int(mask.sum())}
        for r in rate_names:
            row[f"{r}_baseline"] = round(baseline_rates[(model_name, g)][r], 4)
            row[r] = round(rates[r], 4)
            row[f"{r}_change"] = round(rates[r] - baseline_rates[(model_name, g)][r], 4)
        group_rows.append(row)

supporting_table = pd.DataFrame(group_rows)
supporting_table = supporting_table.sort_values(["Group", "Processor", "Model", "ConstrainedOn", "TargetedConstraint"])

os.makedirs("results", exist_ok=True)
supporting_table.to_excel("results/supporting_table_intersectional_groups.xlsx", index=False)

print(f"Total rows: {len(supporting_table)}")
print("Saved to results/supporting_table_intersectional_groups.xlsx")
supporting_table

Total rows: 228
Saved to results/supporting_table_intersectional_groups.xlsx


,Processor,Model,ConstrainedOn,TargetedConstraint,Group,n,SelectionRate_baseline,SelectionRate,SelectionRate_change,TPR_baseline,...,FPR_change,Precision_baseline,Precision,Precision_change,Accuracy_baseline,Accuracy,Accuracy_change,F1_Score_baseline,F1_Score,F1_Score_change
1,Baseline_Balanced,LogisticRegression,n/a,n/a,female_25plus,4810,0.3299,0.3299,0.0000,0.5926,...,0.0000,0.3648,0.3648,0.0000,0.7077,0.7077,0.0000,0.4516,0.4516,0.0000
5,Baseline_Balanced,RandomForest,n/a,n/a,female_25plus,4810,0.1909,0.1909,0.0000,0.4852,...,0.0000,0.5163,0.5163,0.0000,0.8031,0.8031,0.0000,0.5003,0.5003,0.0000
9,Baseline_Balanced,XGBoost,n/a,n/a,female_25plus,4810,0.2474,0.2474,0.0000,0.5527,...,0.0000,0.4538,0.4538,0.0000,0.7740,0.7740,0.0000,0.4984,0.4984,0.0000
29,ExponentiatedGradient,LogisticRegression,age_only,demographic_parity,female_25plus,4810,0.3299,0.2769,-0.0530,0.5926,...,-0.0459,0.3648,0.3754,0.0105,0.7077,0.7279,0.0202,0.4516,0.4331,-0.0186
33,ExponentiatedGradient,LogisticRegression,age_only,equalized_odds,female_25plus,4810,0.3299,0.3383,0.0083,0.5926,...,0.0269,0.3648,0.3171,-0.0477,0.7077,0.6732,-0.0345,0.4516,0.3963,-0.0553
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
226,ThresholdOptimizer,XGBoost,age_only,equalized_odds,male_under25,226,0.4248,0.2566,-0.1681,0.6000,...,-0.1987,0.4688,0.6379,0.1692,0.6416,0.7389,0.0973,0.5263,0.5564,0.0301
210,ThresholdOptimizer,XGBoost,gender_only,demographic_parity,male_under25,226,0.4248,0.2699,-0.1549,0.6000,...,-0.1788,0.4688,0.6066,0.1378,0.6416,0.7257,0.0841,0.5263,0.5441,0.0178
222,ThresholdOptimizer,XGBoost,gender_only,equalized_odds,male_under25,226,0.4248,0.3053,-0.1195,0.6000,...,-0.1325,0.4688,0.5507,0.0820,0.6416,0.6991,0.0575,0.5263,0.5278,0.0015
206,ThresholdOptimizer,XGBoost,intersection,demographic_parity,male_under25,226,0.4248,0.2301,-0.1947,0.6000,...,-0.2185,0.4688,0.6538,0.1851,0.6416,0.7389,0.0973,0.5263,0.5354,0.0091
